In [1]:
%load_ext IPython.extensions.autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

def find_src_folder(current_path: Path, folder_name: str = 'src') -> Path:
    search_directories = [current_path] + list(current_path.parents)
    for parent in search_directories:
        if parent.name == folder_name:
            return parent.parent
    return current_path

src_path = find_src_folder(Path.cwd(), 'src')
sys.path.append(str(src_path))

Dada la información extraída y almacenada en formato .parquet, se realiza un análisis exploratorio sobre la información recolectada

## Importación de librerias necesarias

In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from src.utils import SparkUtils
from pyspark.sql import functions as F, types as T, DataFrame, Window
import json

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\tensorflow_hub\__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


# Resumen de Pasos de Procesamiento de Datos

Este documento resume todos los pasos de procesamiento realizados en los archivos `quality_analysis.ipynb` y `eda.ipynb` para el análisis de reseñas de Amazon.

## 1. Análisis de Calidad de Datos (Quality Analysis)

### 1.1 Análisis de Información de Productos (meta_items)

#### Carga de Datos
- **Fuente**: Tabla Delta `meta_items` con información de productos
- **Columnas disponibles**: title, main_category, features, description, average_rating, rating_number, price, store, parent_asin, categories, details, images

#### Análisis de Valores Nulos
- **Proceso**: Evaluación sistemática de valores nulos por columna
- **Método**: Uso de función `check_null_values()` para cada columna
- **Resultados clave**:
  - `title`, `features`, `description`, `parent_asin`, `categories`, `details`, `images`: 0% nulos
  - `main_category`: 7.4% nulos (231,570 registros)
  - `average_rating`: 0.001% nulos (25 registros)
  - `rating_number`: 0.1% nulos (2,959 registros)
  - `price`: 70.9% nulos (2,215,374 registros)
  - `store`: 1.0% nulos (32,379 registros)

#### Análisis de Listas Vacías
- **Campos evaluados**: features, description, categories, images
- **Método**: Uso de función `check_empty_arrays()` para detectar listas vacías
- **Visualización**: Gráficos de barras porcentuales mostrando distribución de listas vacías vs no vacías

#### Análisis de Campo Details
- **Proceso**: Evaluación de contenido JSON en campo `details`
- **Método**: Función UDF `get_json_length()` para contar elementos en JSON
- **Resultados**: Análisis de productos con/sin detalles especificados
- **Visualización**: Gráfico circular mostrando porcentaje de productos sin detalles

### 1.2 Análisis de Información de Reseñas (reviews)

#### Carga de Datos
- **Fuente**: Tabla Delta `reviews` con información de reseñas
- **Columnas disponibles**: rating, title, text, timestamp, helpful_vote, parent_asin, images

#### Análisis de Valores Nulos
- **Proceso**: Evaluación sistemática de valores nulos por columna
- **Resultados**: Análisis detallado de completitud de datos en reseñas

#### Análisis de Listas de Imágenes
- **Campo evaluado**: images
- **Método**: Detección de listas vacías en campo de imágenes
- **Visualización**: Gráfico circular mostrando distribución de listas vacías vs no vacías

## 2. Análisis Exploratorio de Datos (EDA)

### 2.1 Procesamiento de Texto - Descripciones

#### Limpieza de Texto
- **Proceso**: Unificación y limpieza de descripciones de productos
- **Pasos**:
  1. **Unificación**: Combinación de múltiples descripciones en una sola columna
  2. **Limpieza HTML**: Eliminación de tags HTML usando regex `<[^>]+>`
  3. **Tokenización**: División del texto en palabras individuales
  4. **Eliminación de Stop Words**: Remoción de palabras comunes sin valor semántico

#### Análisis de Longitud de Texto
- **Métricas calculadas**:
  - Longitud mínima, máxima, media y desviación estándar
  - Distribución de longitudes de palabras limpias
- **Filtrado**: Eliminación de registros con descripciones vacías o inválidas
- **Visualización**: Histogramas de distribución de longitudes de texto

### 2.2 Procesamiento de Texto - Títulos

#### Limpieza de Títulos
- **Proceso similar a descripciones**:
  1. **Limpieza HTML**: Eliminación de tags HTML
  2. **Tokenización**: División en palabras
  3. **Eliminación de Stop Words**: Limpieza de palabras comunes
- **Análisis de longitud**: Métricas estadísticas de longitud de títulos
- **Visualización**: Histogramas de distribución de longitudes

### 2.3 Procesamiento de Texto - Características (Features)

#### Limpieza de Features
- **Proceso idéntico a descripciones y títulos**:
  1. **Unificación**: Combinación de múltiples características
  2. **Limpieza HTML**: Eliminación de tags
  3. **Tokenización y limpieza**: Procesamiento de texto estándar
- **Análisis de longitud**: Estadísticas de longitud de características
- **Visualización**: Distribución de longitudes de características

### 2.4 Análisis de Diversidad Léxica (TTR - Type-Token Ratio)

#### Cálculo de TTR
- **Método**: 
  1. **Explosión de tokens**: División de todas las palabras limpias
  2. **Conteo único**: Cálculo de tokens únicos por producto
  3. **Cálculo TTR**: Ratio entre tokens únicos y totales
- **Fórmula**: `TTR = unique_token_count / token_count`
- **Visualización**: Histogramas de distribución de TTR

### 2.5 Análisis de Categorías y Distribución

#### Análisis por Categoría Principal
- **Proceso**:
  1. **Agrupación**: Agrupación por `main_category`
  2. **Conteo de productos**: Número de productos por categoría
  3. **Análisis de distribución**: Porcentaje de productos por categoría
- **Métricas**:
  - Conteo total de productos por categoría
  - Porcentaje de distribución
  - Análisis de categorías más/menos representadas

#### Análisis de Reseñas por Categoría
- **Proceso**:
  1. **Join con reseñas**: Unión de datos de productos y reseñas
  2. **Agrupación**: Agrupación por categoría principal
  3. **Conteo de reseñas**: Número de reseñas por categoría
- **Métricas**:
  - Total de reseñas por categoría
  - Promedio de reseñas por producto
  - Distribución de reseñas

### 2.6 Análisis de Calidad de Datos

#### Evaluación de Completitud
- **Métricas**:
  - Porcentaje de valores nulos por columna
  - Porcentaje de listas vacías
  - Porcentaje de campos con contenido válido
- **Visualización**: Gráficos de barras porcentuales y circulares

#### Análisis de Consistencia
- **Verificación**:
  - Consistencia entre campos relacionados
  - Validación de formatos de datos
  - Detección de anomalías en distribuciones

## 3. Transformaciones de Datos Aplicadas

### 3.1 Limpieza de Texto
- **Eliminación de HTML**: Regex `<[^>]+>` para remover tags
- **Tokenización**: División en palabras usando espacios y puntuación
- **Normalización**: Conversión a minúsculas y eliminación de caracteres especiales
- **Eliminación de Stop Words**: Remoción de palabras comunes del idioma inglés

### 3.2 Filtrado de Datos
- **Filtros aplicados**:
  - Eliminación de registros con texto vacío
  - Eliminación de listas vacías
  - Filtrado por longitud mínima de texto
- **Criterios de calidad**:
  - Texto con al menos una palabra válida
  - Longitud mínima de caracteres
  - Contenido semánticamente relevante

### 3.3 Agregaciones y Agrupaciones
- **Agrupaciones por**:
  - Categoría principal (`main_category`)
  - Producto (`parent_asin`)
  - Características de texto
- **Métricas calculadas**:
  - Conteos totales y únicos
  - Promedios y desviaciones estándar
  - Porcentajes y ratios

### 3.4 Unión de Datasets
- **Joins realizados**:
  - Productos con reseñas por `parent_asin`
  - Datos de texto procesado con metadatos
  - Análisis de categorías con datos de productos

## 4. Almacenamiento de Resultados

### 4.1 Tablas Delta Creadas
- `meta_items_descriptions_unified_words_clean`
- `meta_items_descriptions_unified_words_clean_with_length`
- `meta_items_titles_unified_words_clean`
- `meta_items_titles_unified_words_clean_with_length`
- `meta_items_features_unified_words_clean`
- `meta_items_features_unified_words_clean_with_length`
- `meta_items_texts_words` (datos unificados)
- `meta_items_ttr` (análisis de diversidad léxica)

### 4.2 Formato de Almacenamiento
- **Formato**: Delta Lake para versionado y optimización
- **Particionado**: Por categorías principales cuando es apropiado
- **Compresión**: Optimización automática de Delta Lake
- **Metadatos**: Preservación de esquemas y tipos de datos

## 5. Visualizaciones Generadas

### 5.1 Análisis de Calidad
- Gráficos de barras porcentuales para valores nulos
- Gráficos circulares para distribución de listas vacías
- Tablas resumen de estadísticas de completitud

### 5.2 Análisis de Distribución
- Histogramas de longitudes de texto
- Distribuciones de diversidad léxica (TTR)
- Análisis de frecuencia por categorías

### 5.3 Análisis Comparativo
- Comparación entre categorías principales
- Análisis temporal de tendencias
- Distribución de reseñas por producto

Este procesamiento proporciona una base sólida para análisis posteriores de clustering, análisis de sentimientos y recomendaciones basadas en el contenido de los productos y reseñas de Amazon.

In [4]:
# Configuración de Spark
spark_utils = SparkUtils('preprocessing')
spark = spark_utils.spark

# Carga de datos principales
meta_items = spark.read.format('delta').load(spark_utils.path('meta_items'))
reviews = spark.read.format('delta').load(spark_utils.path('reviews'))

print(f"Productos cargados: {meta_items.count():,}")
print(f"Reseñas cargadas: {reviews.count():,}")
print(f"Columnas de productos: {', '.join(meta_items.columns)}")
print(f"Columnas de reseñas: {', '.join(reviews.columns)}")


Productos cargados: 3,125,022
Reseñas cargadas: 74,204,685
Columnas de productos: title, main_category, features, description, average_rating, rating_number, price, store, parent_asin, categories, details, images
Columnas de reseñas: rating, title, text, timestamp, helpful_vote, parent_asin, images


In [5]:
# Importar funciones de análisis de calidad
from src.utils.dataframe import check_null_values, check_empty_arrays

# Análisis de valores nulos en productos
print("=== ANÁLISIS DE VALORES NULOS - PRODUCTOS ===")
columns_data_nulls = []

for col in meta_items.columns:
    null_count, non_null_count, _ = check_null_values(meta_items, col)
    total = null_count + non_null_count
    null_percentage = (null_count / total) * 100 if total > 0 else 0
    columns_data_nulls.append((col, non_null_count, null_count, null_percentage))
    print(f"{col}: {non_null_count:,} no nulos, {null_count:,} nulos ({null_percentage:.2f}%)")

# Crear DataFrame para visualización
pd_df_columns_data_nulls = pd.DataFrame(
    columns_data_nulls,
    columns=["Column", "Non-null", "Null", "Null_Percentage"]
)
print("\nResumen de valores nulos:")
print(pd_df_columns_data_nulls)


=== ANÁLISIS DE VALORES NULOS - PRODUCTOS ===
title: 3,125,022 no nulos, 0 nulos (0.00%)
main_category: 2,893,452 no nulos, 231,570 nulos (7.41%)
features: 3,125,022 no nulos, 0 nulos (0.00%)
description: 3,125,022 no nulos, 0 nulos (0.00%)
average_rating: 3,124,997 no nulos, 25 nulos (0.00%)
rating_number: 3,122,063 no nulos, 2,959 nulos (0.09%)
price: 909,648 no nulos, 2,215,374 nulos (70.89%)
store: 3,092,643 no nulos, 32,379 nulos (1.04%)
parent_asin: 3,125,022 no nulos, 0 nulos (0.00%)
categories: 3,125,022 no nulos, 0 nulos (0.00%)
details: 3,125,022 no nulos, 0 nulos (0.00%)
images: 3,125,022 no nulos, 0 nulos (0.00%)

Resumen de valores nulos:
            Column  Non-null     Null  Null_Percentage
0            title   3125022        0         0.000000
1    main_category   2893452   231570         7.410188
2         features   3125022        0         0.000000
3      description   3125022        0         0.000000
4   average_rating   3124997       25         0.000800
5    ratin

In [6]:
# Análisis de listas vacías en productos
print("=== ANÁLISIS DE LISTAS VACÍAS - PRODUCTOS ===")
columns_data_empties = []

for col in ["features", "description", "categories", "images"]:
    empty_count, non_empty_count, _ = check_empty_arrays(meta_items, col)
    total = empty_count + non_empty_count
    empty_percentage = (empty_count / total) * 100 if total > 0 else 0
    columns_data_empties.append((col, non_empty_count, empty_count, empty_percentage))
    print(f"{col}: {non_empty_count:,} no vacías, {empty_count:,} vacías ({empty_percentage:.2f}%)")

# Crear DataFrame para visualización
pd_df_columns_data_empties = pd.DataFrame(
    columns_data_empties,
    columns=["Column", "Non-empty", "Empty", "Empty_Percentage"]
)
print("\nResumen de listas vacías:")
print(pd_df_columns_data_empties)


=== ANÁLISIS DE LISTAS VACÍAS - PRODUCTOS ===
features: 2,165,450 no vacías, 959,572 vacías (30.71%)
description: 1,669,143 no vacías, 1,455,879 vacías (46.59%)
categories: 2,664,130 no vacías, 460,892 vacías (14.75%)
images: 3,122,215 no vacías, 2,807 vacías (0.09%)

Resumen de listas vacías:
        Column  Non-empty    Empty  Empty_Percentage
0     features    2165450   959572         30.706088
1  description    1669143  1455879         46.587800
2   categories    2664130   460892         14.748440
3       images    3122215     2807          0.089823


In [7]:
# Análisis de campo details (JSON)
print("=== ANÁLISIS DE CAMPO DETAILS ===")

def get_json_length(str_obj):
    try:
        obj = json.loads(str_obj)
        return len(obj)
    except:
        return 0

# Aplicar función UDF para contar elementos en JSON
meta_items_json_length = (
    meta_items
    .select(
        F.col('*'), 
        F.udf(get_json_length, T.IntegerType())(F.col('details')).alias('len')
    )
)

# Contar productos con/sin detalles
meta_items_details_empty = meta_items_json_length.filter(F.col('len') == 0).count()
meta_items_details_non_empty = meta_items_json_length.filter(F.col('len') > 0).count()
total_items = meta_items_json_length.count()

print(f"Productos con detalles: {meta_items_details_non_empty:,} ({(meta_items_details_non_empty/total_items)*100:.2f}%)")
print(f"Productos sin detalles: {meta_items_details_empty:,} ({(meta_items_details_empty/total_items)*100:.2f}%)")
print(f"Total de productos: {total_items:,}")


=== ANÁLISIS DE CAMPO DETAILS ===
Productos con detalles: 3,080,259 (98.57%)
Productos sin detalles: 44,763 (1.43%)
Total de productos: 3,125,022


In [8]:
# Análisis de valores nulos en reseñas
print("=== ANÁLISIS DE VALORES NULOS - RESEÑAS ===")
columns_data_nulls_reviews = []

for col in reviews.columns:
    null_count, non_null_count, _ = check_null_values(reviews, col)
    total = null_count + non_null_count
    null_percentage = (null_count / total) * 100 if total > 0 else 0
    columns_data_nulls_reviews.append((col, non_null_count, null_count, null_percentage))
    print(f"{col}: {non_null_count:,} no nulos, {null_count:,} nulos ({null_percentage:.2f}%)")

# Crear DataFrame para visualización
pd_df_columns_data_nulls_reviews = pd.DataFrame(
    columns_data_nulls_reviews,
    columns=["Column", "Non-null", "Null", "Null_Percentage"]
)
print("\nResumen de valores nulos en reseñas:")
print(pd_df_columns_data_nulls_reviews)


=== ANÁLISIS DE VALORES NULOS - RESEÑAS ===
rating: 74,204,685 no nulos, 0 nulos (0.00%)
title: 74,204,685 no nulos, 0 nulos (0.00%)
text: 74,204,685 no nulos, 0 nulos (0.00%)
timestamp: 74,204,685 no nulos, 0 nulos (0.00%)
helpful_vote: 74,204,685 no nulos, 0 nulos (0.00%)
parent_asin: 74,204,685 no nulos, 0 nulos (0.00%)
images: 74,204,685 no nulos, 0 nulos (0.00%)

Resumen de valores nulos en reseñas:
         Column  Non-null  Null  Null_Percentage
0        rating  74204685     0              0.0
1         title  74204685     0              0.0
2          text  74204685     0              0.0
3     timestamp  74204685     0              0.0
4  helpful_vote  74204685     0              0.0
5   parent_asin  74204685     0              0.0
6        images  74204685     0              0.0


In [9]:
# Importar librerías de procesamiento de texto
from pyspark.ml.feature import Tokenizer, StopWordsRemover

# 1. Unificación de descripciones
print("=== PROCESAMIENTO DE DESCRIPCIONES ===")
meta_items_descriptions_unified = meta_items.withColumn(
    "description_colapsed",
    F.concat_ws(" ", F.col("description"))
)

# 2. Limpieza de HTML
meta_items_descriptions_unified_clean = meta_items_descriptions_unified.withColumn(
    "description_colapsed_clean",
    F.regexp_replace(F.col("description_colapsed"), "<[^>]+>", " ")
)

# 3. Tokenización
tokenizer = Tokenizer(inputCol="description_colapsed_clean", outputCol="words")
tokenized = tokenizer.transform(meta_items_descriptions_unified_clean)

# 4. Eliminación de Stop Words
remover = StopWordsRemover(inputCol="words", outputCol="clean_words")
meta_items_descriptions_unified_words_clean = remover.transform(tokenized)

print("Descripciones procesadas exitosamente")
print(f"Registros procesados: {meta_items_descriptions_unified_words_clean.count():,}")


=== PROCESAMIENTO DE DESCRIPCIONES ===
Descripciones procesadas exitosamente
Registros procesados: 3,125,022


In [10]:
# Análisis de longitud de descripciones
print("=== ANÁLISIS DE LONGITUD DE DESCRIPCIONES ===")

# Agregar columna de longitud
meta_items_descriptions_unified_words_clean_with_length = (
    meta_items_descriptions_unified_words_clean
    .withColumn('clean_words_length', F.size(F.col('clean_words')))
    .filter(
        (F.col('clean_words_length') > 0) &
        (F.col('clean_words')[0] != F.lit(""))
    )
)

# Calcular estadísticas
meta_items_descriptions_unified_words_clean_with_length_count = (
    meta_items_descriptions_unified_words_clean_with_length
    .agg(
        F.min('clean_words_length').alias('min_words_length'),
        F.max('clean_words_length').alias('max_words_length'),
        F.mean('clean_words_length').alias('mean_words_length'),
        F.std('clean_words_length').alias('std_words_length'),
    )
    .collect()[0]
)

# Mostrar estadísticas
stats = {
    "min": meta_items_descriptions_unified_words_clean_with_length_count[0],
    "max": meta_items_descriptions_unified_words_clean_with_length_count[1],
    "mean": meta_items_descriptions_unified_words_clean_with_length_count[2],
    "std": meta_items_descriptions_unified_words_clean_with_length_count[3],
}

print(f"Longitud mínima: {stats['min']}")
print(f"Longitud máxima: {stats['max']}")
print(f"Longitud promedio: {stats['mean']:.2f}")
print(f"Desviación estándar: {stats['std']:.2f}")
print(f"Registros válidos: {meta_items_descriptions_unified_words_clean_with_length.count():,}")


=== ANÁLISIS DE LONGITUD DE DESCRIPCIONES ===
Longitud mínima: 1
Longitud máxima: 63839
Longitud promedio: 89.02
Desviación estándar: 148.12
Registros válidos: 1,668,566


In [11]:
# Procesamiento de títulos
print("=== PROCESAMIENTO DE TÍTULOS ===")

# 1. Limpieza de HTML en títulos
meta_items_titles_unified_clean = meta_items.withColumn(
    "title_clean",
    F.regexp_replace(F.col("title"), "<[^>]+>", " ")
)

# 2. Tokenización de títulos
tokenizer_titles = Tokenizer(inputCol="title_clean", outputCol="words")
tokenized_titles = tokenizer_titles.transform(meta_items_titles_unified_clean)

# 3. Eliminación de Stop Words en títulos
remover_titles = StopWordsRemover(inputCol="words", outputCol="clean_words")
meta_items_titles_unified_words_clean = remover_titles.transform(tokenized_titles)

# 4. Análisis de longitud de títulos
meta_items_titles_unified_words_clean_with_length = (
    meta_items_titles_unified_words_clean
    .withColumn('clean_words_length', F.size(F.col('clean_words')))
    .filter(
        (F.col('clean_words_length') > 0) &
        (F.col('clean_words')[0] != F.lit(""))
    )
)

# Calcular estadísticas de títulos
titles_stats = (
    meta_items_titles_unified_words_clean_with_length
    .agg(
        F.min('clean_words_length').alias('min_words_length'),
        F.max('clean_words_length').alias('max_words_length'),
        F.mean('clean_words_length').alias('mean_words_length'),
        F.std('clean_words_length').alias('std_words_length'),
    )
    .collect()[0]
)

print(f"Títulos procesados: {meta_items_titles_unified_words_clean_with_length.count():,}")
print(f"Longitud mínima: {titles_stats[0]}")
print(f"Longitud máxima: {titles_stats[1]}")
print(f"Longitud promedio: {titles_stats[2]:.2f}")
print(f"Desviación estándar: {titles_stats[3]:.2f}")


=== PROCESAMIENTO DE TÍTULOS ===
Títulos procesados: 3,124,740
Longitud mínima: 1
Longitud máxima: 370
Longitud promedio: 18.58
Desviación estándar: 8.56


In [12]:
# Procesamiento de características (features)
print("=== PROCESAMIENTO DE CARACTERÍSTICAS ===")

# 1. Unificación de características
meta_items_features_unified = meta_items.withColumn(
    "features_colapsed",
    F.concat_ws(" ", F.col("features"))
)

# 2. Limpieza de HTML en características
meta_items_features_unified_clean = meta_items_features_unified.withColumn(
    "features_colapsed_clean",
    F.regexp_replace(F.col("features_colapsed"), "<[^>]+>", " ")
)

# 3. Tokenización de características
tokenizer_features = Tokenizer(inputCol="features_colapsed_clean", outputCol="words")
tokenized_features = tokenizer_features.transform(meta_items_features_unified_clean)

# 4. Eliminación de Stop Words en características
remover_features = StopWordsRemover(inputCol="words", outputCol="clean_words")
meta_items_features_unified_words_clean = remover_features.transform(tokenized_features)

# 5. Análisis de longitud de características
meta_items_features_unified_words_clean_with_length = (
    meta_items_features_unified_words_clean
    .withColumn('clean_words_length', F.size(F.col('clean_words')))
    .filter(
        (F.col('clean_words_length') > 0) &
        (F.col('clean_words')[0] != F.lit(""))
    )
)

# Calcular estadísticas de características
features_stats = (
    meta_items_features_unified_words_clean_with_length
    .agg(
        F.min('clean_words_length').alias('min_words_length'),
        F.max('clean_words_length').alias('max_words_length'),
        F.mean('clean_words_length').alias('mean_words_length'),
        F.std('clean_words_length').alias('std_words_length'),
    )
    .collect()[0]
)

print(f"Características procesadas: {meta_items_features_unified_words_clean_with_length.count():,}")
print(f"Longitud mínima: {features_stats[0]}")
print(f"Longitud máxima: {features_stats[1]}")
print(f"Longitud promedio: {features_stats[2]:.2f}")
print(f"Desviación estándar: {features_stats[3]:.2f}")


=== PROCESAMIENTO DE CARACTERÍSTICAS ===
Características procesadas: 2,160,568
Longitud mínima: 1
Longitud máxima: 1316
Longitud promedio: 73.36
Desviación estándar: 54.23


In [13]:
# Unificación de todos los textos procesados
print("=== ANÁLISIS DE DIVERSIDAD LÉXICA (TTR) ===")

# Unir descripciones, títulos y características
meta_items_texts_words = (
    meta_items_descriptions_unified_words_clean
    .join(
        meta_items_titles_unified_words_clean.select("parent_asin", F.col("clean_words").alias("title_words")),
        on="parent_asin", how="left"
    )
    .join(
        meta_items_features_unified_words_clean.select("parent_asin", F.col("clean_words").alias("features_words")),
        on="parent_asin", how="left"
    )
    .select(
        F.col("parent_asin"),
        F.col("clean_words").alias("description_words"),
        F.col("title_words"),
        F.col("features_words")
    )
)

# Calcular TTR (Type-Token Ratio)
exploded = meta_items_texts_words.withColumn("token", F.explode("description_words"))
unique_counts = exploded.groupBy("parent_asin").agg(F.countDistinct("token").alias("unique_token_count"))
token_counts = meta_items_texts_words.withColumn("token_count", F.size("description_words"))

meta_items_ttr = token_counts.join(unique_counts, on="parent_asin", how="inner") \
    .withColumn("TTR", F.col("unique_token_count") / F.col("token_count"))

# Calcular estadísticas de TTR
ttr_stats = (
    meta_items_ttr
    .agg(
        F.min('TTR').alias('min_ttr'),
        F.max('TTR').alias('max_ttr'),
        F.mean('TTR').alias('mean_ttr'),
        F.std('TTR').alias('std_ttr'),
    )
    .collect()[0]
)

print(f"Productos con TTR calculado: {meta_items_ttr.count():,}")
print(f"TTR mínimo: {ttr_stats[0]:.4f}")
print(f"TTR máximo: {ttr_stats[1]:.4f}")
print(f"TTR promedio: {ttr_stats[2]:.4f}")
print(f"Desviación estándar TTR: {ttr_stats[3]:.4f}")


=== ANÁLISIS DE DIVERSIDAD LÉXICA (TTR) ===
Productos con TTR calculado: 3,124,872
TTR mínimo: 0.0025
TTR máximo: 1.0000
TTR promedio: 0.9140
Desviación estándar TTR: 0.1267


In [14]:
# Análisis de distribución por categorías
print("=== ANÁLISIS DE CATEGORÍAS ===")

# Análisis de productos por categoría
category_analysis = (
    meta_items
    .groupBy('main_category')
    .agg(F.count('*').alias('product_count'))
    .withColumn('total_products', F.lit(meta_items.count()))
    .withColumn('category_percentage', F.col('product_count') / F.col('total_products') * 100)
    .orderBy(F.desc('product_count'))
)

print("Top 10 categorías por número de productos:")
category_analysis.limit(10).show(truncate=False)

# Análisis de reseñas por categoría
reviews_by_category = (
    reviews
    .join(meta_items.select('parent_asin', 'main_category'), on='parent_asin', how='inner')
    .groupBy('main_category')
    .agg(F.count('*').alias('review_count'))
    .withColumn('total_reviews', F.lit(reviews.count()))
    .withColumn('review_percentage', F.col('review_count') / F.col('total_reviews') * 100)
    .orderBy(F.desc('review_count'))
)

print("\nTop 10 categorías por número de reseñas:")
reviews_by_category.limit(10).show(truncate=False)

# Análisis combinado
combined_analysis = (
    category_analysis
    .join(reviews_by_category, on='main_category', how='inner')
    .withColumn('avg_reviews_per_product', F.col('review_count') / F.col('product_count'))
    .orderBy(F.desc('product_count'))
)

print("\nAnálisis combinado (productos y reseñas por categoría):")
combined_analysis.limit(10).show(truncate=False)


=== ANÁLISIS DE CATEGORÍAS ===
Top 10 categorías por número de productos:
+-------------------------+-------------+--------------+-------------------+
|main_category            |product_count|total_products|category_percentage|
+-------------------------+-------------+--------------+-------------------+
|Cell Phones & Accessories|1187766      |3125022       |38.00824442195927  |
|Computers                |444322       |3125022       |14.218203903844518 |
|All Electronics          |439060       |3125022       |14.049821089259531 |
|NULL                     |231570       |3125022       |7.410187832277661  |
|Camera & Photo           |229474       |3125022       |7.343116304461217  |
|Home Audio & Theater     |109955       |3125022       |3.518535229511984  |
|Video Games              |82756        |3125022       |2.6481733568595676 |
|Appstore for Android     |68679        |3125022       |2.197712528103802  |
|Industrial & Scientific  |56999        |3125022       |1.823955159355678  |
|A

In [ ]:
# Guardar resultados procesados en formato Delta
print("=== GUARDANDO RESULTADOS PROCESADOS ===")

# Guardar descripciones procesadas
print("Guardando descripciones procesadas...")
meta_items_descriptions_unified_words_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .save(spark_utils.path('meta_items_descriptions_unified_words_clean'))

meta_items_descriptions_unified_words_clean_with_length.write \
    .format("delta") \
    .mode("overwrite") \
    .save(spark_utils.path('meta_items_descriptions_unified_words_clean_with_length'))

# Guardar títulos procesados
print("Guardando títulos procesados...")
meta_items_titles_unified_words_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .save(spark_utils.path('meta_items_titles_unified_words_clean'))

meta_items_titles_unified_words_clean_with_length.write \
    .format("delta") \
    .mode("overwrite") \
    .save(spark_utils.path('meta_items_titles_unified_words_clean_with_length'))

# Guardar características procesadas
print("Guardando características procesadas...")
meta_items_features_unified_words_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .save(spark_utils.path('meta_items_features_unified_words_clean'))

meta_items_features_unified_words_clean_with_length.write \
    .format("delta") \
    .mode("overwrite") \
    .save(spark_utils.path('meta_items_features_unified_words_clean_with_length'))

# Guardar textos unificados y TTR
print("Guardando textos unificados y análisis TTR...")
meta_items_texts_words.write \
    .format("delta") \
    .mode("overwrite") \
    .save(spark_utils.path('meta_items_texts_words'))

meta_items_ttr.write \
    .format("delta") \
    .mode("overwrite") \
    .save(spark_utils.path('meta_items_ttr'))

print("✅ Todos los resultados han sido guardados exitosamente en formato Delta Lake")


=== GUARDANDO RESULTADOS PROCESADOS ===
Guardando descripciones procesadas...
Guardando títulos procesados...


In [ ]:
# Resumen final de todo el procesamiento realizado
print("=" * 60)
print("RESUMEN FINAL DEL PROCESAMIENTO DE DATOS")
print("=" * 60)

print(f"\n📊 DATOS ORIGINALES:")
print(f"   • Productos: {meta_items.count():,}")
print(f"   • Reseñas: {reviews.count():,}")

print(f"\n🔍 ANÁLISIS DE CALIDAD:")
print(f"   • Columnas analizadas en productos: {len(meta_items.columns)}")
print(f"   • Columnas analizadas en reseñas: {len(reviews.columns)}")
print(f"   • Campos con listas vacías evaluados: 4 (features, description, categories, images)")

print(f"\n📝 PROCESAMIENTO DE TEXTO:")
print(f"   • Descripciones procesadas: {meta_items_descriptions_unified_words_clean_with_length.count():,}")
print(f"   • Títulos procesados: {meta_items_titles_unified_words_clean_with_length.count():,}")
print(f"   • Características procesadas: {meta_items_features_unified_words_clean_with_length.count():,}")

print(f"\n📈 ANÁLISIS DE DIVERSIDAD LÉXICA:")
print(f"   • Productos con TTR calculado: {meta_items_ttr.count():,}")
print(f"   • TTR promedio: {ttr_stats[2]:.4f}")

print(f"\n📁 ARCHIVOS GENERADOS:")
print(f"   • meta_items_descriptions_unified_words_clean")
print(f"   • meta_items_descriptions_unified_words_clean_with_length")
print(f"   • meta_items_titles_unified_words_clean")
print(f"   • meta_items_titles_unified_words_clean_with_length")
print(f"   • meta_items_features_unified_words_clean")
print(f"   • meta_items_features_unified_words_clean_with_length")
print(f"   • meta_items_texts_words")
print(f"   • meta_items_ttr")

print(f"\n✅ PROCESAMIENTO COMPLETADO EXITOSAMENTE")
print(f"   Todos los datos han sido procesados y almacenados en formato Delta Lake")
print(f"   Los datos están listos para análisis posteriores de clustering y ML")
print("=" * 60)
